In [1]:
!pip install sentence-transformers torch


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
from sentence_transformers import SentenceTransformer, util
import torch

# 1. 디바이스 설정 - M4 MacBook은 MPS 사용
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")

# 2. 모델 로드
model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS', device=device)

# 3. 검색 대상 데이터베이스 (Corpus)
documents = [
    "AI 모델의 파인튜닝은 특정 도메인 데이터로 성능을 최적화하는 과정입니다.",
    "RAG는 외부 지식을 참조하여 LLM의 환각 현상을 줄여주는 기술입니다.",
    "BERT는 양방향 문맥 이해에 특화된 인코더 모델입니다.",
    "맥북 프로 M3 Max는 고성능 AI 추론에 적합한 통합 메모리를 갖추고 있습니다.",
    "RunPod은 클라우드 환경에서 GPU 자원을 대여해주는 서비스입니다."
]

# 4. 문서 임베딩 - MPS에서는 convert_to_tensor=True 대신 numpy로 변환 후 처리
#    MPS 백엔드에서 cos_sim 연산 시 호환성 문제가 발생할 수 있으므로 CPU 텐서로 변환
document_embeddings = model.encode(documents, convert_to_tensor=False)
document_embeddings = torch.tensor(document_embeddings)

# 5. 사용자 질문 (Query) 및 임베딩
query = "AI 모델을 특정 분야에 맞게 학습시키고 싶어"
query_embedding = model.encode(query, convert_to_tensor=False)
query_embedding = torch.tensor(query_embedding)

# 6. 유사도 계산 (Cosine Similarity) - CPU 텐서에서 수행
cosine_scores = util.cos_sim(query_embedding, document_embeddings)[0]

# 7. 결과 출력 (상위 2개 추출)
top_results = torch.topk(cosine_scores, k=2)
print(f"질문: {query}\n")
print("-" * 30)
for score, idx in zip(top_results.values, top_results.indices):
    print(f"유사도 점수: {score:.4f}")
    print(f"검색된 문서: {documents[idx]}\n")

ModuleNotFoundError: Could not import module 'PreTrainedModel'. Are this object's requirements defined correctly?